# Predict next word with an LSTM

The bigram feed-forward model in `predict_next_word_simple_nn.ipynb` only sees **one** word at a time.

With these sentences:
Look at the right-hand column on its own. `not` is asked to predict both `bad` and `good`; so is `is`. The bigram model has no way to tell the cases apart — it can only learn the average of the two.

An LSTM carries a hidden state across the sequence, so by the time it reads `not` it still remembers whether the sentence started with `moana1` or `moana2`.

Note the data is deliberately laid out so the first word alone is *not* enough either — `moana1` appears with both `not` and `is`. Only the **combination** of both words determines the answer.

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.tokenize import word_tokenize

In [17]:
sentences = [
    "moana1 is good",
    "moana2 is bad",
    "moana1 not bad",
    "moana2 not good"
]

In [18]:
# ------------------
# Preprocessing
# ------------------

sentence = " ".join(sentences)

tokens = word_tokenize(sentence)
vocab = sorted(list(set(tokens)))

word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

# Instead of bigram pairs, keep each sentence as one sequence.
# Input  = all words except the last
# Target = all words except the first (shifted by one)
input_sequences = []
target_sequences = []

for s in sentences:
    words = s.split()
    indices = [word_to_idx[w] for w in words]

    input_sequences.append(indices[:-1])
    target_sequences.append(indices[1:])

print("Tokens:     ", tokens)
print("Vocab:      ", vocab)
print("Word to idx:", word_to_idx)
print("Inputs:     ", input_sequences)
print("Targets:    ", target_sequences)



Tokens:      ['moana1', 'is', 'good', 'moana2', 'is', 'bad', 'moana1', 'not', 'bad', 'moana2', 'not', 'good']
Vocab:       ['bad', 'good', 'is', 'moana1', 'moana2', 'not']
Word to idx: {'bad': 0, 'good': 1, 'is': 2, 'moana1': 3, 'moana2': 4, 'not': 5}
Inputs:      [[3, 2], [4, 2], [3, 5], [4, 5]]
Targets:     [[2, 1], [2, 0], [5, 0], [5, 1]]


In [19]:
# ------------------
# Model Layers
# ------------------

embedding = nn.Embedding(len(vocab), 8)
lstm = nn.LSTM(8, 16, batch_first=True)
output = nn.Linear(16, len(vocab))

params = (
    list(embedding.parameters())
    + list(lstm.parameters())
    + list(output.parameters())
)

optimizer = optim.Adam(params, lr=0.01)
loss_fn = nn.CrossEntropyLoss()

In [20]:
# ------------------
# Training
# ------------------

X = torch.tensor(input_sequences)   # (batch, seq_len)
y = torch.tensor(target_sequences)  # (batch, seq_len)

for epoch in range(500):

    x = embedding(X)                # (batch, seq_len, 8)
    x, _ = lstm(x)                  # (batch, seq_len, 16)
    logits = output(x)              # (batch, seq_len, vocab)

    # CrossEntropyLoss wants (N, classes) and (N,), so flatten batch+time
    loss = loss_fn(logits.reshape(-1, len(vocab)), y.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch {epoch + 1:3d}  loss {loss.item():.4f}")

epoch 100  loss 0.3574
epoch 200  loss 0.3490
epoch 300  loss 0.3478
epoch 400  loss 0.3473
epoch 500  loss 0.3471


In [21]:
# ------------------
# Prediction
# ------------------

def predict(context):
    """context is a string of one or more words, e.g. "moana1 not"."""
    words = context.split()
    idx = torch.tensor([[word_to_idx[w] for w in words]])

    x = embedding(idx)
    x, _ = lstm(x)
    logits = output(x)

    # Only the last timestep predicts the next word
    pred = torch.argmax(logits[0, -1]).item()

    return idx_to_word[pred]

# Same last word "not", different first word -> different answer
print("moana1 not ->", predict("moana1 not"))
print("moana2 not ->", predict("moana2 not"))

# Same again for "is"
print("moana1 is  ->", predict("moana1 is"))
print("moana2 is  ->", predict("moana2 is"))

# Ambiguous on purpose: both "not" and "is" follow each movie in the data,
# so whatever comes out here is an arbitrary tie-break, not a learned fact.
print("moana1     ->", predict("moana1"))
print("moana2     ->", predict("moana2"))

moana1 not -> bad
moana2 not -> good
moana1 is  -> good
moana2 is  -> bad
moana1     -> not
moana2     -> not


## What to notice

All four two-word prefixes resolve correctly:

```
moana1 not -> bad     moana1 is -> good
moana2 not -> good    moana2 is -> bad
```

This is an XOR-shaped problem. Neither word alone carries the answer:

- fix the second word to `not` and the answer still flips with the first word
- fix the first word to `moana1` and the answer still flips with the second word

The model has to combine both, and the LSTM's hidden state is where that combination lives. A bigram model, seeing only the most recent token, is structurally incapable of it — no amount of training fixes that.

The single-word predictions are a genuine coin flip here, since `moana1` is followed by `not` in one sentence and `is` in another. Whichever word wins is an artifact of initialization, not something the model learned.

### Caveat

Four sentences and a six-word vocab mean the loss goes near zero by memorization. This shows the architecture *can* represent the dependency — it is not evidence it would learn a real negation pattern from natural text. That would need many examples and a held-out test set.